# Netflix Recommendation Engine

In [4]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [5]:
ratings_path = "../data/ratings.txt"
movies_path = "../data/movies.csv"

ratings = []
current_movie_id = None

with open(ratings_path, "r", encoding="latin1") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if line.endswith(":"):
            current_movie_id = int(line[:-1])
        else:
            user_id, rating, _ = line.split(",")
            ratings.append([current_movie_id, int(user_id), int(rating)])

ratings = pd.DataFrame(ratings, columns=["movieId", "userId", "rating"])

movies = pd.read_csv(movies_path)

df = ratings.merge(movies, on="movieId", how="inner")

print(df.head())
print(df.shape)

   movieId   userId  rating             title  \
0        1  1488844       3  Toy Story (1995)   
1        1   822109       5  Toy Story (1995)   
2        1   885013       4  Toy Story (1995)   
3        1    30878       4  Toy Story (1995)   
4        1   823519       3  Toy Story (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1  Adventure|Animation|Children|Comedy|Fantasy  
2  Adventure|Animation|Children|Comedy|Fantasy  
3  Adventure|Animation|Children|Comedy|Fantasy  
4  Adventure|Animation|Children|Comedy|Fantasy  
(23813503, 5)


## 1. Most Popular and Liked Genre

In [6]:
genre_df = df.assign(
    genre=df["genres"].str.split("|")
).explode("genre")

genre_stats = (
    genre_df.groupby("genre")
    .agg(
        total_ratings=("rating", "count"),
        average_rating=("rating", "mean")
    )
    .sort_values("total_ratings", ascending=False)
)

print("Most popular genre:")
print(genre_stats["total_ratings"].idxmax())
print(genre_stats.loc[genre_stats["total_ratings"].idxmax(), "total_ratings"])

print("\nMost liked genre:")
print(genre_stats["average_rating"].idxmax())
print(genre_stats.loc[genre_stats["average_rating"].idxmax(), "average_rating"])

Most popular genre:
Drama
11450961

Most liked genre:
IMAX
3.8263583259982394


## 2. Best-Suited Movie for One User in Every Genre

In [7]:
user_id = 1235099

movie_counts = ratings["movieId"].value_counts()
valid_movies = movie_counts[movie_counts >= 100].index

ratings_model = ratings[
    ratings["movieId"].isin(valid_movies)
].copy()

movie_ids = ratings_model["movieId"].unique()
user_ids = ratings_model["userId"].unique()

movie_to_idx = {movie_id: i for i, movie_id in enumerate(movie_ids)}
user_to_idx = {user_id: i for i, user_id in enumerate(user_ids)}

rows = ratings_model["movieId"].map(movie_to_idx)
cols = ratings_model["userId"].map(user_to_idx)

movie_user_matrix = csr_matrix(
    (ratings_model["rating"], (rows, cols)),
    shape=(len(movie_ids), len(user_ids))
)

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21
)

model.fit(movie_user_matrix)

user_ratings = ratings_model[
    ratings_model["userId"] == user_id
].copy()

liked_movies = user_ratings[
    user_ratings["rating"] >= 4
].sort_values("rating", ascending=False)

print("Movies rated 4 or 5:", len(liked_movies))

Movies rated 4 or 5: 223


In [8]:
recommendations = {}
rated_movie_ids = set(user_ratings["movieId"])

for _, row in liked_movies.iterrows():
    movie_id = row["movieId"]
    user_rating = row["rating"]

    if movie_id not in movie_to_idx:
        continue

    distances, indices = model.kneighbors(
        movie_user_matrix[movie_to_idx[movie_id]],
        n_neighbors=21
    )

    for distance, neighbor_idx in zip(distances[0][1:], indices[0][1:]):
        neighbor_movie_id = movie_ids[neighbor_idx]

        if neighbor_movie_id in rated_movie_ids:
            continue

        similarity = 1 - distance
        score = similarity * user_rating

        if neighbor_movie_id not in recommendations:
            recommendations[neighbor_movie_id] = []

        recommendations[neighbor_movie_id].append(score)

predictions = pd.DataFrame([
    {
        "movieId": movie_id,
        "predicted_score": np.mean(scores)
    }
    for movie_id, scores in recommendations.items()
])

recommendations_df = predictions.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

recommendation_genres = (
    recommendations_df
    .assign(genre=recommendations_df["genres"].str.split("|"))
    .explode("genre")
)

best_recommendations = (
    recommendation_genres
    .sort_values("predicted_score", ascending=False)
    .groupby("genre")
    .first()[["title", "predicted_score"]]
    .sort_index()
)

print(best_recommendations)

                                                         title  \
genre                                                            
Action                    Adventures of Robin Hood, The (1938)   
Adventure                 Adventures of Robin Hood, The (1938)   
Animation                            Swan Princess, The (1994)   
Children                             Swan Princess, The (1994)   
Comedy                                     Fletch Lives (1989)   
Crime                                      Marathon Man (1976)   
Documentary     Line King: The Al Hirschfeld Story, The (1996)   
Drama               Central Station (Central do Brasil) (1998)   
Fantasy                      House II: The Second Story (1987)   
Film-Noir                        Sweet Smell of Success (1957)   
Horror       Texas Chainsaw Massacre: The Next Generation (...   
IMAX                                   Wings of Courage (1995)   
Musical                             Blues Brothers, The (1980)   
Mystery   

## 3. Best and Worst-Rated Genres

In [9]:
best_genre = genre_stats["average_rating"].idxmax()
worst_genre = genre_stats["average_rating"].idxmin()

print("Best-rated genre:", best_genre)
print("Average rating:", genre_stats.loc[best_genre, "average_rating"])

print("Worst-rated genre:", worst_genre)
print("Average rating:", genre_stats.loc[worst_genre, "average_rating"])

Best-rated genre: IMAX
Average rating: 3.8263583259982394
Worst-rated genre: Documentary
Average rating: 3.5137625712181118
